# E31 S4百分比与配对复核

## 结果摘要
按附件的异常值标记剔除2行后为26人；先泪水15人、先对照11人。原表配对/比率可重现。先泪水组相对均值差52.08%，未重现图注73.6%。

## 范围与假设
仅核图2D-E/S4公开数值；不重跑显著性或神经影像，不验证实验记录来源和因果。没有新增剔除规则。相对均值差定义为100×(对照均值−泪水均值)/对照均值，区别于绝对差、个体百分比平均。

## 数据
来源：https://journals.plos.org/plosbiology/article/file?id=10.1371%2Fjournal.pbio.3002442.s001&type=supplementary
下载2026-10-29，39,459字节。原表不随笔记分发。运行前将附件命名agron2023_S1Data.xlsx置于工作目录；安装openpyxl。下列代码校验SHA-256，仅输出聚合结果。

## 计算与结果


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib, json, statistics
import openpyxl
source = Path("agron2023_S1Data.xlsx")  # Download the publisher S1 Data beside this notebook.
assert hashlib.sha256(source.read_bytes()).hexdigest() == "a05ed6573b488585803e2ddb254e756fcc40f33def41507ebc0557a69569ae20"
w = openpyxl.load_workbook(source, data_only=True, read_only=True)
a = [r for r in list(w["Fig 2D-E"].values)[1:] if r[0] is not None]
b = [r for r in list(w["S4A-S4C Fig"].values)[1:] if r[0] is not None]
bmap = {r[0]: r for r in b}
valid = [r for r in a if r[10] != "outlier"]
result = {
    "rows": len(a), "unique_ids": len({r[0] for r in a}),
    "flagged_outliers": sum(r[10] == "outlier" for r in a),
    "required_numeric_missing": sum(any(r[i] is None for i in [1,3,4,6,7,8]) for r in a),
    "nonpositive_provocations": sum(r[i] <= 0 for r in a for i in [3,6]),
    "apr_max_ratio_error": max(max(abs(r[1]/r[3]-r[7]), abs(r[4]/r[6]-r[8])) for r in a),
    "s4_ids_match": set(bmap) == {r[0] for r in a},
    "s4_day_max_mapping_error": max(max(abs(bmap[r[0]][1]-(r[7] if r[9]=="Tears" else r[8])), abs(bmap[r[0]][2]-(r[8] if r[9]=="Tears" else r[7]))) for r in a),
    "group_summary": {}
}
for label, rows in [("all", valid), ("tears_first", [r for r in valid if r[9]=="Tears"]), ("saline_first", [r for r in valid if r[9]=="Saline"])]:
    t, s = statistics.mean(r[7] for r in rows), statistics.mean(r[8] for r in rows)
    result["group_summary"][label] = {"n":len(rows), "mean_tears":t, "mean_saline":s, "absolute_s_minus_t":s-t, "relative_percent_vs_saline":100*(s-t)/s}
result["mean_day2_minus_day1"] = statistics.mean(bmap[r[0]][2]-bmap[r[0]][1] for r in valid)
result["tears_first_zero_saline_denominators"] = sum(r[8] == 0 for r in valid if r[9]=="Tears")
print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "rows": 28,
  "unique_ids": 28,
  "flagged_outliers": 2,
  "required_numeric_missing": 0,
  "nonpositive_provocations": 0,
  "apr_max_ratio_error": 0.0,
  "s4_ids_match": true,
  "s4_day_max_mapping_error": 0,
  "group_summary": {
    "all": {
      "n": 26,
      "mean_tears": 0.9670845410041854,
      "mean_saline": 1.3057747399659165,
      "absolute_s_minus_t": 0.33869019896173114,
      "relative_percent_vs_saline": 25.9378734015464
    },
    "tears_first": {
      "n": 15,
      "mean_tears": 0.6773806221746725,
      "mean_saline": 1.4134518150694622,
      "absolute_s_minus_t": 0.7360711928947896,
      "relative_percent_vs_saline": 52.07614331434541
    },
    "saline_first": {
      "n": 11,
      "mean_tears": 1.3621353394080666,
      "mean_saline": 1.158942364824718,
      "absolute_s_minus_t": -0.20319297458334873,
      "relative_percent_vs_saline": -17.532621185530676
    }
  },
  "mean_day2_minus_day1": 0.5106227159168724,
  "tears_first_zero_saline_denominators":

### 相邻图表的分母核对
只沿用各表异常值标记，不从工作表总行数直接当分析人数；图4/图5右侧另列内容不与左侧逐行混接。


In [ ]:
from collections import Counter
def data_rows(sheet):
    return [r for r in list(w[sheet].values)[1:] if r[0] is not None]
exp1_all = data_rows("Fig 2A-C")
exp1 = [r for r in exp1_all if r[10] != "outlier"]
mean_t = statistics.mean(r[7] for r in exp1)
mean_s = statistics.mean(r[8] for r in exp1)
behavior = {r[0]:r for r in valid}
brain_all = data_rows("Fig 4B-C")
brain = [r for r in brain_all if r[10] != "outlier"]
connect_all = data_rows("Fig 5B-D")
connect = [r for r in connect_all if r[5] != "outlier"]
balanced = data_rows("S6 Fig")
checks = {
 "experiment1": {"source_rows":len(exp1_all), "included":len(exp1), "unique_ids":len({r[0] for r in exp1_all}), "mean_tears":mean_t, "mean_saline":mean_s, "relative_percent":100*(mean_s-mean_t)/mean_s,
   "ratio_max_error":max(max(abs(r[1]/r[3]-r[7]),abs(r[4]/r[6]-r[8])) for r in exp1_all)},
 "fig4": {"rows_with_outliers":len(brain_all),"included":len(brain), "all_ids_match_behavior":all(r[0] in behavior for r in brain),
   "apr_max_error":max(max(abs(r[7]-behavior[r[0]][8]),abs(r[8]-behavior[r[0]][7])) for r in brain)},
 "fig5": {"rows_with_outliers":len(connect_all),"included":len(connect), "all_ids_match_behavior":all(r[0] in behavior for r in connect)},
 "s6": {"included":len(balanced), "first_day_counts":dict(Counter(r[1] for r in balanced)),
   "apr_diff_max_error":max(abs(r[4]-(behavior[r[0]][8]-behavior[r[0]][7])) for r in balanced)},
 "fig1_odor_counts_by_experiment":dict(Counter(r[8] for r in data_rows("Fig 1A-C")))
}
print(json.dumps(checks,ensure_ascii=False,indent=2))


{
  "experiment1": {
    "source_rows": 26,
    "included": 25,
    "unique_ids": 26,
    "mean_tears": 0.93323782532,
    "mean_saline": 1.67205107484,
    "relative_percent": 44.1860455483214,
    "ratio_max_error": 4.705882350464208e-10
  },
  "fig4": {
    "rows_with_outliers": 26,
    "included": 24,
    "all_ids_match_behavior": true,
    "apr_max_error": 4.440892098500626e-15
  },
  "fig5": {
    "rows_with_outliers": 26,
    "included": 24,
    "all_ids_match_behavior": true
  },
  "s6": {
    "included": 18,
    "first_day_counts": {
      "Tears": 9,
      "Saline": 9
    },
    "apr_diff_max_error": 6.217248937900877e-15
  },
  "fig1_odor_counts_by_experiment": {
    "1": 22,
    "3": 24
  }
}


## 解释与验证状态
绝对均值差为0.736071 APR，乘100不自动成为百分比；未获得作者的73.6%计算公式。15人中6人的对照值为零，因此不能不加定义地求所有人的相对变化均值。

2026-10-29已用普通Python解释器从干净命名空间顺序运行代码，输出为该次实际捕获；JSON结构、代码语法、输出聚合值已检查。当前环境没有nbformat/nbclient/ipykernel/nbconvert，未完成Jupyter内核运行及界面视觉检查。execution_count保留空值以区分普通Python运行。可安装上述包，在数据所在目录执行 `jupyter nbconvert --execute --to notebook --inplace <本笔记绝对路径>` 后打开核对。

相邻图表复核：实验1附件的25人均值为0.933238/1.672051，均值相对差44.19%，未重现正文43.7%；图4/5剔附件异常标记后各24行，S6为9＋9＝18行。未重跑相关或显著性。
